In [0]:
spark.conf.set(
    "fs.azure.account.key.datalake0501.dfs.core.windows.net",
    "YOUR_STORAGE_ACCOUNT_KEY_HERE"  # Replace with your key
)

from pyspark.sql import functions as F

fact_orders = spark.read.format("delta").load(
    "abfss://silver@datalake0501.dfs.core.windows.net/fact_orders/"
)
dim_customer = spark.read.format("delta").load(
    "abfss://silver@datalake0501.dfs.core.windows.net/dim_customer/"
)
dim_product = spark.read.format("delta").load(
    "abfss://silver@datalake0501.dfs.core.windows.net/dim_product/"
)

print("Loaded successfully")

In [0]:
gold_revenue = fact_orders.join(
    dim_customer.select('customer_id', 'country', 'customer_segment'),
    on='customer_id', how='left'
).join(
    dim_product.select('product_id', 'category', 'sub_category', 'brand'),
    on='product_id', how='left'
).groupBy('country', 'category', 'customer_segment', 'order_year', 'order_month') \
 .agg(
     F.count('order_id').alias('total_orders'),
     F.round(F.sum('total_price_usd'), 2).alias('total_revenue'),
     F.round(F.sum('profit_usd'), 2).alias('total_profit'),
     F.round(F.avg('profit_margin'), 2).alias('avg_profit_margin'),
     F.sum(F.when(F.col('high_value_order_flag') == True, 1).otherwise(0)).alias('high_value_orders'),
     F.sum(F.when(F.col('fraud_flag') == True, 1).otherwise(0)).alias('fraud_orders')
 )

gold_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("order_year", "order_month") \
    .save("abfss://gold@datalake0501.dfs.core.windows.net/revenue_by_country_category/")

print(f"Revenue table written: {gold_revenue.count()} rows")

In [0]:
gold_monthly = fact_orders.groupBy('order_year', 'order_month', 'order_week') \
    .agg(
        F.count('order_id').alias('total_orders'),
        F.round(F.sum('total_price_usd'), 2).alias('total_revenue'),
        F.round(F.sum('profit_usd'), 2).alias('total_profit'),
        F.round(F.avg('profit_margin'), 2).alias('avg_profit_margin'),
        F.countDistinct('customer_id').alias('unique_customers'),
        F.sum(F.when(F.col('fraud_flag') == True, 1).otherwise(0)).alias('fraud_orders')
    ).orderBy('order_year', 'order_month', 'order_week')

gold_monthly.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("order_year", "order_month") \
    .save("abfss://gold@datalake0501.dfs.core.windows.net/monthly_sales_trends/")

print(f"Monthly trends written: {gold_monthly.count()} rows")

In [0]:
gold_customers = fact_orders.join(
    dim_customer.select('customer_id', 'country', 'customer_segment', 'age', 'gender'),
    on='customer_id', how='left'
).groupBy('customer_segment', 'country', 'gender') \
 .agg(
     F.countDistinct('customer_id').alias('unique_customers'),
     F.count('order_id').alias('total_orders'),
     F.round(F.avg('total_price_usd'), 2).alias('avg_order_value'),
     F.round(F.sum('total_price_usd'), 2).alias('total_revenue'),
     F.round(F.avg('profit_margin'), 2).alias('avg_profit_margin'),
     F.round(F.avg('age'), 1).alias('avg_age')
 )

gold_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://gold@datalake0501.dfs.core.windows.net/customer_segment_analysis/")

print(f"Customer analysis written: {gold_customers.count()} rows")

In [0]:
gold_fraud = fact_orders.join(
    dim_customer.select('customer_id', 'country', 'customer_segment'),
    on='customer_id', how='left'
).groupBy('country', 'customer_segment', 'order_year', 'order_month') \
 .agg(
     F.count('order_id').alias('total_orders'),
     F.sum(F.when(F.col('fraud_flag') == True, 1).otherwise(0)).alias('fraud_orders'),
     F.round(F.avg('fraud_risk_score'), 2).alias('avg_fraud_risk_score'),
     F.round(
         F.sum(F.when(F.col('fraud_flag') == True, 1).otherwise(0)) / F.count('order_id') * 100, 2
     ).alias('fraud_rate_percent')
 )

gold_fraud.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("order_year", "order_month") \
    .save("abfss://gold@datalake0501.dfs.core.windows.net/fraud_risk_summary/")

print(f"Fraud summary written: {gold_fraud.count()} rows")